In [20]:
import os, glob

print("Mounted datasets:", os.listdir("/kaggle/input"))

# list parquet files across all mounted datasets
parquets = glob.glob("/kaggle/input/**/*.parquet", recursive=True)
print("Parquet files found:", parquets)
print("Count:", len(parquets))

Mounted datasets: ['datasets']
Parquet files found: ['/kaggle/input/datasets/shinyhoro/gluons-and-sparks/QCDToGGQQ_IMGjet_RH1all_jet0_run2_n55494_LR.parquet', '/kaggle/input/datasets/shinyhoro/lr-and-sr-gluons-and-sparks/QCDToGGQQ_IMGjet_RH1all_jet0_run1_n47540_LR.parquet', '/kaggle/input/datasets/shinyhoro/lr-and-sr-gluons-and-sparks/QCDToGGQQ_IMGjet_RH1all_jet0_run0_n36272_LR.parquet']
Count: 3


In [21]:
!pip -q install pyarrow pandas numpy tqdm

import pyarrow.parquet as pq

PATH = parquets[0]  # pick run0 first if multiple
pf = pq.ParquetFile(PATH)

print("Using:", PATH)
print("Row groups:", pf.num_row_groups)
print("Top-level columns:", pf.schema_arrow.names)

Using: /kaggle/input/datasets/shinyhoro/gluons-and-sparks/QCDToGGQQ_IMGjet_RH1all_jet0_run2_n55494_LR.parquet
Row groups: 1
Top-level columns: ['X_jets_LR', 'X_jets', 'pt', 'm0', 'y']


In [22]:
import glob
parquets = sorted(glob.glob("/kaggle/input/**/*.parquet", recursive=True))
print("Found", len(parquets), "parquet files:")
for p in parquets:
    print(" -", p)

Found 3 parquet files:
 - /kaggle/input/datasets/shinyhoro/gluons-and-sparks/QCDToGGQQ_IMGjet_RH1all_jet0_run2_n55494_LR.parquet
 - /kaggle/input/datasets/shinyhoro/lr-and-sr-gluons-and-sparks/QCDToGGQQ_IMGjet_RH1all_jet0_run0_n36272_LR.parquet
 - /kaggle/input/datasets/shinyhoro/lr-and-sr-gluons-and-sparks/QCDToGGQQ_IMGjet_RH1all_jet0_run1_n47540_LR.parquet


In [23]:
import glob
import hashlib
import numpy as np
import pyarrow.parquet as pq
import torch
from torch.utils.data import IterableDataset

# ---------------- normalization + padding ----------------

PAD = (1,2,1,2)

def pad_hr_to_128(x):
    t,b,l,r = PAD
    return np.pad(x, ((0,0),(t,b),(l,r)), mode="constant", constant_values=0.0)

def crop_128_to_125(x):
    t,b,l,r = PAD
    return x[:,t:128-b,l:128-r]

def normalize(x, cap=300.0):
    x = np.clip(x,0,cap)
    return np.log1p(x) / np.log1p(cap)

# ---------------- deterministic split ----------------

def split_of(file_path,row_id,seed=42):

    key = f"{seed}|{file_path}|{row_id}".encode("utf-8")
    h = hashlib.md5(key).hexdigest()

    bucket = int(h[:8],16) % 100

    if bucket < 80:
        return "train"
    elif bucket < 90:
        return "val"
    else:
        return "test"

# ---------------- dataset ----------------

class CMSMultiParquetSRIterable(IterableDataset):

    def __init__(self, parquet_paths, split,
                 cap=300.0,
                 batch_size=4,
                 seed=42,
                 max_samples=None):

        assert split in ("train","val","test")

        self.paths = list(parquet_paths)
        self.split = split
        self.cap = cap
        self.batch_size = batch_size
        self.seed = seed
        self.max_samples = max_samples


    def __iter__(self):

        cols = ["X_jets_LR","X_jets","pt","m0","y"]

        yielded = 0

        for path in self.paths:

            pf = pq.ParquetFile(path)
            row_id = 0

            for rb in pf.iter_batches(batch_size=self.batch_size,columns=cols):

                d = rb.to_pydict()
                n = len(d["y"])

                for i in range(n):

                    if split_of(path,row_id,self.seed) != self.split:
                        row_id += 1
                        continue

                    lr = np.array(d["X_jets_LR"][i],dtype=np.float32)
                    hr = np.array(d["X_jets"][i],dtype=np.float32)

                    lr = normalize(lr,self.cap)
                    hr = normalize(hr,self.cap)

                    hr128 = pad_hr_to_128(hr)

                    y  = int(d["y"][i])
                    pt = float(d["pt"][i])
                    m0 = float(d["m0"][i])

                    yield (
                        torch.from_numpy(lr).float(),
                        torch.from_numpy(hr128).float(),
                        torch.tensor(y,dtype=torch.long),
                        torch.tensor(pt,dtype=torch.float32),
                        torch.tensor(m0,dtype=torch.float32),
                    )

                    yielded += 1
                    row_id += 1

                    if self.max_samples and yielded >= self.max_samples:
                        return

In [24]:
#Starting building SRGAN from here
import torch
print(torch.cuda.is_available())

True


In [27]:
import torch
import torch.nn as nn

class SimpleSRGenerator(nn.Module):
    # (B,3,64,64) -> (B,3,128,128)
    def __init__(self, in_ch=3, base=48, blocks=3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_ch, base, 3, 1, 1),
            nn.PReLU()
        )
        body = []
        for _ in range(blocks):
            body += [
                nn.Conv2d(base, base, 3, 1, 1),
                nn.PReLU(),
                nn.Conv2d(base, base, 3, 1, 1),
            ]
        self.body = nn.Sequential(*body)
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="nearest"),  # 64->128
            nn.Conv2d(base, base, 3, 1, 1),
            nn.PReLU(),
            nn.Conv2d(base, 3, 3, 1, 1)
        )

    def forward(self, x):
        h = self.head(x)
        b = self.body(h) + h
        return self.up(b)

In [28]:
import torchvision.utils as vutils
from torchvision.utils import make_grid
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os
from torch.utils.data import DataLoader

WORK_DIR = "/kaggle/working"

CHECKPOINT_DIR = os.path.join(WORK_DIR, "baseline_checkpoints")
IMG_DIR = os.path.join(WORK_DIR, "baseline_sr_debug")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)


device = torch.device("cuda")
G = SimpleSRGenerator().to(device)

opt = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.9, 0.999))
scaler = torch.cuda.amp.GradScaler()

# -------------------------
# DATASET
# -------------------------
train_ds = CMSMultiParquetSRIterable(
    parquets,
    split="train",
    cap=300.0,
    batch_size=2,
    max_samples=None
)

train_dl = DataLoader(train_ds, batch_size=8, num_workers=0, pin_memory=True)

def l1_loss(pred, target):
    return torch.mean(torch.abs(pred - target))

G.train()

it = iter(train_dl)

def get_batch():
    global it
    try:
        return next(it)
    except StopIteration:
        it = iter(train_dl)
        return next(it)

# -------------------------
# TRAINING CONFIG
# -------------------------
steps = 2000         # increase training
log_every = 50
save_every = 500

accum_steps = 4

# -------------------------
# Start Fresh
# -------------------------
start_step = 0  # Starting from step 1



# -------------------------
# TRAINING LOOP
# -------------------------
for step in range(start_step + 1, steps + 1):

    lr, hr128, y, pt, m0 = get_batch()

    lr = lr.to(device, non_blocking=True)
    hr128 = hr128.to(device, non_blocking=True)

    with torch.cuda.amp.autocast():
        sr128 = G(lr)
        loss = l1_loss(sr128, hr128) / accum_steps

    scaler.scale(loss).backward()

    if step % accum_steps == 0:
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)

    if step % log_every == 0:
        print(f"step {step}/{steps}  L1={(loss.item()*accum_steps):.5f}")
        
    # Save and display images every 'save_every' 
    if step % save_every == 0:

    # save checkpoint
        torch.save({
            "step": step,
            "G": G.state_dict(),
            "opt": opt.state_dict()
        }, f"{CHECKPOINT_DIR}/baseline_G_step{step}.pt")

    # save latest checkpoint
        torch.save({
            "step": step,
            "G": G.state_dict(),
            "opt": opt.state_dict()
        }, f"{CHECKPOINT_DIR}/latest.pt")

        print(f"Saved checkpoint at step {step}")

        with torch.no_grad():
            print("saving preview")

            lr_vis = lr[:2]
            sr_vis = sr128[:2]
            hr_vis = hr128[:2]

            sr_resized = F.interpolate(sr_vis, size=(lr_vis.shape[2], lr_vis.shape[3]))
            hr_resized = F.interpolate(hr_vis, size=(lr_vis.shape[2], lr_vis.shape[3]))

            imgs = torch.cat([lr_vis, sr_resized, hr_resized], dim=0)

            vutils.save_image(
                imgs,
                f"{IMG_DIR}/step_{step}.png",
                nrow=2,
                normalize=True
            )

            print(f"Saved SR image: step_{step}.png")
   

/tmp/ipykernel_55/2811825935.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_55/2811825935.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


step 50/2000  L1=0.00599
step 100/2000  L1=0.00264
step 150/2000  L1=0.00249
step 200/2000  L1=0.00201
step 250/2000  L1=0.00147
step 300/2000  L1=0.00164
step 350/2000  L1=0.00163
step 400/2000  L1=0.00162
step 450/2000  L1=0.00167
step 500/2000  L1=0.00160
Saved checkpoint at step 500
saving preview
Saved SR image: step_500.png
step 550/2000  L1=0.00158
step 600/2000  L1=0.00155
step 650/2000  L1=0.00164
step 700/2000  L1=0.00156
step 750/2000  L1=0.00157
step 800/2000  L1=0.00158
step 850/2000  L1=0.00143
step 900/2000  L1=0.00136
step 950/2000  L1=0.00147
step 1000/2000  L1=0.00160
Saved checkpoint at step 1000
saving preview
Saved SR image: step_1000.png
step 1050/2000  L1=0.00158
step 1100/2000  L1=0.00178
step 1150/2000  L1=0.00146
step 1200/2000  L1=0.00167
step 1250/2000  L1=0.00157
step 1300/2000  L1=0.00155
step 1350/2000  L1=0.00127
step 1400/2000  L1=0.00131
step 1450/2000  L1=0.00144
step 1500/2000  L1=0.00144
Saved checkpoint at step 1500
saving preview
Saved SR image: s

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
print(os.listdir("/kaggle/working/baseline_checkpoints"))
print(os.listdir("/kaggle/working/baseline_sr_debug"))

In [ ]:
torch.save(G.state_dict(), "/kaggle/working/sr_baseline_generator.pth")